<a href="https://colab.research.google.com/github/Manarsenic/EDA/blob/main/EDA_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
from io import StringIO
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from scipy.stats import zscore

# Scrape and load data
url = "https://ourworldindata.org/grapher/prevalence-of-undernourishment-in-developing-countries-since-1970.csv?v=1&csvType=full&useColumnShortNames=true"
try:
    resp = requests.get(url)
    resp.raise_for_status()
    df = pd.read_csv(StringIO(resp.text))
    print("Data successfully scraped!")
except requests.exceptions.RequestException as e:
    print(f"Error scraping data: {e}")
    exit()

df.info()
print("\nData Preview:\n",)
df.head()

Data successfully scraped!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 4 columns):
 #   Column                                                                                          Non-Null Count  Dtype  
---  ------                                                                                          --------------  -----  
 0   Entity                                                                                          27 non-null     object 
 1   Code                                                                                            0 non-null      float64
 2   Year                                                                                            27 non-null     int64  
 3   Prevalence of undernourishment in developing countries - FAO (Food Security Indicators) (2017)  27 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 996.0+ bytes

Data Preview:



,Entity,Code,Year,Prevalence of undernourishment in developing countries - FAO (Food Security Indicators) (2017)
0,Developing countries,NaN,1970,34.75
1,Developing countries,NaN,1980,26.50
2,Developing countries,NaN,1991,23.30
3,Developing countries,NaN,1992,23.10
4,Developing countries,NaN,1993,22.70


In [2]:
# Rename columns
df.rename(columns={'Entity': 'country',
                   'Year': 'year',
                   'Prevalence of undernourishment in developing countries - FAO (Food Security Indicators) (2017)': 'und_rate'},
                   inplace=True)
# Change 'year' to integer type
df['year'] = df['year'].astype(int)

df.info()
print("\nDataFrame Preview:\n", df.head())
df.head(19)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   country   27 non-null     object 
 1   Code      0 non-null      float64
 2   year      27 non-null     int64  
 3   und_rate  27 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 996.0+ bytes

DataFrame Preview:
                 country  Code  year  und_rate
0  Developing countries   NaN  1970     34.75
1  Developing countries   NaN  1980     26.50
2  Developing countries   NaN  1991     23.30
3  Developing countries   NaN  1992     23.10
4  Developing countries   NaN  1993     22.70


,country,Code,year,und_rate
0,Developing countries,NaN,1970,34.75
1,Developing countries,NaN,1980,26.50
2,Developing countries,NaN,1991,23.30
3,Developing countries,NaN,1992,23.10
4,Developing countries,NaN,1993,22.70
5,Developing countries,NaN,1994,22.10
6,Developing countries,NaN,1995,21.20
7,Developing countries,NaN,1996,20.40
8,Developing countries,NaN,1997,19.70
9,Developing countries,NaN,1998,19.10


In [3]:
# Introduce missing values for demonstration
idx = df.sample(n=3).index
df.loc[idx, 'und_rate'] = np.nan

# Impute using kNN
imputer = KNNImputer(n_neighbors=5)
imputed_data = imputer.fit_transform(df[['und_rate']])
df['und_rate_imputed'] = imputed_data

print("\nOriginal and Imputed Rates:\n", df[['und_rate', 'und_rate_imputed']].head(25))
df.head(10)


Original and Imputed Rates:
     und_rate  und_rate_imputed
0      34.75         34.750000
1      26.50         26.500000
2        NaN         18.822917
3      23.10         23.100000
4      22.70         22.700000
5      22.10         22.100000
6      21.20         21.200000
7      20.40         20.400000
8      19.70         19.700000
9      19.10         19.100000
10     18.60         18.600000
11       NaN         18.822917
12     18.20         18.200000
13     18.20         18.200000
14     18.30         18.300000
15     18.20         18.200000
16     17.90         17.900000
17     17.30         17.300000
18     16.50         16.500000
19     15.70         15.700000
20     15.00         15.000000
21     14.50         14.500000
22     14.10         14.100000
23     13.70         13.700000
24       NaN         18.822917


,country,Code,year,und_rate,und_rate_imputed
0,Developing countries,NaN,1970,34.75,34.750000
1,Developing countries,NaN,1980,26.50,26.500000
2,Developing countries,NaN,1991,NaN,18.822917
3,Developing countries,NaN,1992,23.10,23.100000
4,Developing countries,NaN,1993,22.70,22.700000
5,Developing countries,NaN,1994,22.10,22.100000
6,Developing countries,NaN,1995,21.20,21.200000
7,Developing countries,NaN,1996,20.40,20.400000
8,Developing countries,NaN,1997,19.70,19.700000
9,Developing countries,NaN,1998,19.10,19.100000


In [4]:
# Label Encoding
le = LabelEncoder()
df['country_encoded'] = le.fit_transform(df['country'])

# One-Hot Encoding
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_cols = ohe.fit_transform(df[['country']])
encoded_df = pd.DataFrame(encoded_cols, columns=ohe.get_feature_names_out(['country']))

print("\nLabel Encoding Preview:\n", df[['country', 'country_encoded']].head())
print("\nOne-Hot Encoding Preview:\n", encoded_df.head())
df.head()



Label Encoding Preview:
                 country  country_encoded
0  Developing countries                0
1  Developing countries                0
2  Developing countries                0
3  Developing countries                0
4  Developing countries                0

One-Hot Encoding Preview:
    country_Developing countries
0                           1.0
1                           1.0
2                           1.0
3                           1.0
4                           1.0


,country,Code,year,und_rate,und_rate_imputed,country_encoded
0,Developing countries,NaN,1970,34.75,34.750000,0
1,Developing countries,NaN,1980,26.50,26.500000,0
2,Developing countries,NaN,1991,NaN,18.822917,0
3,Developing countries,NaN,1992,23.10,23.100000,0
4,Developing countries,NaN,1993,22.70,22.700000,0


In [5]:
# Outlier detection using Z-score
df['z_score'] = zscore(df['und_rate_imputed'])
outliers = df[(df['z_score'] > 3) | (df['z_score'] < -3)]

print("\nDetected Outliers:\n")
if not outliers.empty:
    print(outliers)
else:
    print("No outliers detected.")

# Normalization using Min-Max scaling
scaler = MinMaxScaler()
df['und_rate_norm'] = scaler.fit_transform(df[['und_rate_imputed']])

print("\nNormalized Data Preview:\n", df.head())


Detected Outliers:

                country  Code  year  und_rate  und_rate_imputed  \
0  Developing countries   NaN  1970     34.75             34.75   

   country_encoded   z_score  
0                0  3.577506  

Normalized Data Preview:
                 country  Code  year  und_rate  und_rate_imputed  \
0  Developing countries   NaN  1970     34.75         34.750000   
1  Developing countries   NaN  1980     26.50         26.500000   
2  Developing countries   NaN  1991       NaN         18.822917   
3  Developing countries   NaN  1992     23.10         23.100000   
4  Developing countries   NaN  1993     22.70         22.700000   

   country_encoded   z_score  und_rate_norm  
0                0  3.577506       1.000000  
1                0  1.724410       0.622426  
2                0  0.000000       0.271072  
3                0  0.960709       0.466819  
4                0  0.870862       0.448513  


BASIC EDA PLOTS